# Phase 3.1 & 3.2 — Word2Vec, Embedding Baselines & Limitations

This notebook contains all exploratory experiments for **Phase 3.1** (Word2Vec, CBOW vs. Skip-gram, Vector Arithmetic, and the 5 Limitations of Static Word Embeddings) and the **Step 3.2 Baseline** (TF-IDF Weighted Word2Vec vs. Unweighted Word2Vec vs. Raw TF-IDF).

### Contents
1. **Environment Setup & Imports**
2. **Extracting Clean Prose Sentences from Research Papers** (*Attention Is All You Need* & *BERT*)
3. **Training Word2Vec Models** (CBOW `sg=0` vs. Skip-gram `sg=1`)
4. **Semantic Similarity & Nearest Neighbors**
5. **Vector Arithmetic & Analogies** (e.g. $v_{\text{decoder}} - v_{\text{output}} + v_{\text{input}}$)
6. **Empirical Demonstration of Word2Vec Limitations** (Polysemy, OOV, Word Order Invariance, Negation Blindness)
7. **Step 3.2 Baseline: TF-IDF Weighted Word2Vec** vs. Unweighted Average vs. Sparse TF-IDF

## 1. Environment Setup & Imports

In [1]:
import sys
from pathlib import Path
import numpy as np

# Ensure backend directory is in sys.path when running from notebooks/
project_root = Path("..").resolve()
backend_dir = project_root / "backend"
if str(backend_dir) not in sys.path:
    sys.path.insert(0, str(backend_dir))

from app.models.document import Document
from app.services.parser.pdf_parser import parse_pdf
from app.services.embeddings.word2vec import Word2VecPipeline
from app.services.nlp.tfidf import TFIDFModel
from app.services.nlp.preprocessing import clean_paper_text, tokenize

print("✓ Successfully imported backend services!")

✓ Successfully imported backend services!


## 2. Extract Clean Prose Sentences from Research Papers

We parse *Attention Is All You Need* and *BERT* from `data/papers/`, clean formatting/citations using Phase 2 preprocessors, and extract prose sentences using our spaCy sentence validator.

In [2]:
papers_dir = project_root / "data" / "papers"
pdf_files = sorted(list(papers_dir.glob("*.pdf")))
print(f"Found {len(pdf_files)} PDF files:")
for p in pdf_files:
    print(f" - {p.name}")

docs = [parse_pdf(p) for p in pdf_files]
sentences = Word2VecPipeline.extract_sentences_from_documents(docs)
total_tokens = sum(len(s) for s in sentences)

print(f"\nTotal Clean Sentences Extracted: {len(sentences):,}")
print(f"Total Tokens in Paper Corpus:   {total_tokens:,}")
print(f"Example Sentence 1:             {' '.join(sentences[0])}")
print(f"Example Sentence 2:             {' '.join(sentences[1])}")

Found 2 PDF files:
 - Attention-is-all-you-need.pdf
 - BERT.pdf

Total Clean Sentences Extracted: 500
Total Tokens in Paper Corpus:   9,674
Example Sentence 1:             attention is all you need
Example Sentence 2:             the dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and decoder


## 3. Training Word2Vec Models (CBOW vs. Skip-gram)

- **CBOW (`sg=0`)**: Continuous Bag-of-Words predicts target word from context average. Fast and smooth for common words.
- **Skip-gram (`sg=1`)**: Predicts context words from target word. Slower, but better for rare technical terms.

In [3]:
# 1. Train CBOW
cbow_pipeline = Word2VecPipeline(
    vector_size=100,
    window=5,
    min_count=2,
    sg=0,  # CBOW
    epochs=60,
    seed=42,
)
cbow_pipeline.train(sentences)

# 2. Train Skip-gram
skipgram_pipeline = Word2VecPipeline(
    vector_size=100,
    window=5,
    min_count=2,
    sg=1,  # Skip-gram
    epochs=60,
    seed=42,
)
skipgram_pipeline.train(sentences)

print(f"CBOW Vocabulary Size:      {cbow_pipeline.vocab_size} words")
print(f"Skip-gram Vocabulary Size: {skipgram_pipeline.vocab_size} words")

CBOW Vocabulary Size:      922 words
Skip-gram Vocabulary Size: 922 words


## 4. Semantic Similarity & Nearest Neighbors

Let us measure the cosine similarity between word pairs across both models.

In [4]:
test_pairs = [
    ("encoder", "decoder"),
    ("attention", "recurrent"),
    ("bert", "transformer"),
    ("layer", "layers"),
    ("training", "gpu"),
    ("model", "attention"),
    ("random", "transformer"),
]

print(f"{'Word Pair':<28} | {'CBOW':<10} | {'Skip-gram':<10}")
print("-" * 54)
for w1, w2 in test_pairs:
    if cbow_pipeline.has_word(w1) and cbow_pipeline.has_word(w2):
        s_cbow = cbow_pipeline.similarity(w1, w2)
        s_sg = skipgram_pipeline.similarity(w1, w2)
        print(f"{w1 + ' <-> ' + w2:<28} | {s_cbow:+.4f}    | {s_sg:+.4f}")

Word Pair                    | CBOW       | Skip-gram 
------------------------------------------------------
encoder <-> decoder          | +0.9779    | +0.8554
attention <-> recurrent      | +0.8360    | +0.4617
bert <-> transformer         | +0.6158    | +0.3651
layer <-> layers             | +0.8893    | +0.5380
training <-> gpu             | +0.4588    | +0.2670
model <-> attention          | +0.0966    | +0.0989
random <-> transformer       | +0.0183    | -0.0261


In [5]:
# Probe top nearest neighbors for key technical concepts
probe_terms = ["attention", "transformer", "recurrent", "layer"]
for term in probe_terms:
    if cbow_pipeline.has_word(term):
        nn = cbow_pipeline.most_similar(positive=[term], topn=4)
        nn_str = ", ".join(f"{w} ({score:.2f})" for w, score in nn)
        print(f"Nearest to '{term}': {nn_str}")

Nearest to 'attention': mechanism (0.97), head (0.95), self (0.95), multi (0.94)
Nearest to 'transformer': uses (0.84), ture (0.79), architecture (0.77), former (0.75)
Nearest to 'recurrent': convolutional (0.89), mechanism (0.89), wise (0.85), mechanisms (0.85)
Nearest to 'layer': sub (0.97), followed (0.94), parallel (0.94), composed (0.93)


## 5. Vector Arithmetic & Analogies

Testing vector offset algebra on scientific concepts:  
$$v(\text{"decoder"}) - v(\text{"output"}) + v(\text{"input"}) \approx ?$$

In [6]:
try:
    analogy = cbow_pipeline.most_similar(
        positive=["decoder", "input"],
        negative=["output"],
        topn=4,
    )
    print("Vector Arithmetic: decoder - output + input:")
    for rank, (w, s) in enumerate(analogy, 1):
        print(f"  {rank}. {w:<16} (score: {s:.4f})")
except KeyError as e:
    print(f"Could not run analogy: {e}")

Vector Arithmetic: decoder - output + input:
  1. encoder          (score: 0.9289)
  2. position         (score: 0.9062)
  3. prevent          (score: 0.8975)
  4. attend           (score: 0.8915)


## 6. Empirical Demonstration of Word2Vec Limitations

Here we test the 4 critical failure modes of static word embeddings:
1. **Polysemy**: Same word used in different contexts gets the exact same static vector.
2. **Out-of-Vocabulary (OOV)**: Unseen words or typos crash with KeyError.
3. **Word Order Invariance**: Inverting subject and object produces an identical vector ($1.000000$).
4. **Negation Blindness**: Adding "not" cannot flip the truth polarity.

In [7]:
print("=== LIMITATION 1: POLYSEMY / CONTEXT BLINDNESS ===")
# 'head' in 'multi-head attention' vs 'injury to the head'
if cbow_pipeline.has_word("head"):
    vec_head = cbow_pipeline.get_vector("head")
    print(f"Static vector for 'head' has shape {vec_head.shape} and norm {np.linalg.norm(vec_head):.2f}")
    print("Word2Vec assigns this EXACT same vector regardless of whether 'head' means a neural network block or anatomy.")

print("\n=== LIMITATION 2: OUT-OF-VOCABULARY (OOV) ===")
oov_words = ["roberta", "deberta", "attension"]
for w in oov_words:
    try:
        cbow_pipeline.get_vector(w)
    except KeyError as e:
        print(f"Caught expected OOV error: {e}")

print("\n=== LIMITATION 3: WORD ORDER INVARIANCE ===")
s1 = "model uses attention instead of recurrent layers"
s2 = "model uses recurrent layers instead of attention"
sim_order = cbow_pipeline.sentence_similarity(s1, s2)
print(f"S1: '{s1}'")
print(f"S2: '{s2}'")
print(f"Cosine Similarity: {sim_order:.6f}  <-- Mathematically identical due to commutative addition!")

print("\n=== LIMITATION 4: NEGATION BLINDNESS ===")
s3 = "this transformer architecture is effective"
s4 = "this transformer architecture is not effective"
sim_neg = cbow_pipeline.sentence_similarity(s3, s4)
print(f"S3: '{s3}'")
print(f"S4: '{s4}'")
print(f"Cosine Similarity: {sim_neg:.6f}  <-- Still ~0.95+ despite opposite meaning!")

=== LIMITATION 1: POLYSEMY / CONTEXT BLINDNESS ===
Static vector for 'head' has shape (100,) and norm 3.79
Word2Vec assigns this EXACT same vector regardless of whether 'head' means a neural network block or anatomy.

=== LIMITATION 2: OUT-OF-VOCABULARY (OOV) ===
Caught expected OOV error: "Word 'roberta' is Out-Of-Vocabulary (OOV). Vocabulary size: 922 terms."
Caught expected OOV error: "Word 'deberta' is Out-Of-Vocabulary (OOV). Vocabulary size: 922 terms."
Caught expected OOV error: "Word 'attension' is Out-Of-Vocabulary (OOV). Vocabulary size: 922 terms."

=== LIMITATION 3: WORD ORDER INVARIANCE ===
S1: 'model uses attention instead of recurrent layers'
S2: 'model uses recurrent layers instead of attention'
Cosine Similarity: 1.000000  <-- Mathematically identical due to commutative addition!

=== LIMITATION 4: NEGATION BLINDNESS ===
S3: 'this transformer architecture is effective'
S4: 'this transformer architecture is not effective'
Cosine Similarity: 0.978472  <-- Still ~0.95+ de

## 7. Step 3.2 Baseline: TF-IDF Weighted Word2Vec

Now we compare all three representation methods side-by-side:
1. **Raw Sparse TF-IDF** (Phase 2)
2. **Unweighted Word2Vec Average** (Phase 3.1)
3. **TF-IDF Weighted Word2Vec Average** (Our Step 3.2 Baseline)

$$\vec{v}_{\text{sentence}} = \frac{\sum_{t} \text{TF-IDF}(t) \cdot \vec{v}(t)}{\sum_{t} \text{TF-IDF}(t)}$$

In [8]:
# Fit Phase 2 TF-IDF model on corpus sentences
tfidf_model = TFIDFModel(smooth=True).fit(sentences)
print(f"✓ TFIDFModel fitted with {len(tfidf_model.vocab):,} terms.")

def sparse_cosine(d1, d2):
    common = set(d1.keys()) & set(d2.keys())
    if not common:
        return 0.0
    dot = sum(d1[k] * d2[k] for k in common)
    n1 = np.sqrt(sum(v ** 2 for v in d1.values()))
    n2 = np.sqrt(sum(v ** 2 for v in d2.values()))
    return float(dot / (n1 * n2)) if n1 > 0 and n2 > 0 else 0.0

comparison_pairs = [
    (
        "Related Concepts (Transformer & Attention)",
        "the transformer relies entirely on an self attention mechanism without recurrence",
        "multi head attention connects the encoder and decoder representations",
    ),
    (
        "Shared Common Filler Words, Different Topics",
        "we trained the deep model on eight gpus for three days",
        "the bert language model was evaluated on eleven downstream nlp tasks",
    ),
    (
        "Word Order Inversion",
        "model uses attention instead of recurrent layers",
        "model uses recurrent layers instead of attention",
    ),
]

for label, sa, sb in comparison_pairs:
    print(f"\n{'='*75}")
    print(f"Test Case: {label}")
    print(f"  Sentence A: '{sa}'")
    print(f"  Sentence B: '{sb}'")
    
    # Method 1: Sparse TF-IDF
    ta = [t.lower() for t in tokenize(clean_paper_text(sa))]
    tb = [t.lower() for t in tokenize(clean_paper_text(sb))]
    sim_sparse = sparse_cosine(tfidf_model.transform(ta), tfidf_model.transform(tb))
    
    # Method 2: Unweighted Word2Vec
    sim_unweighted = cbow_pipeline.sentence_similarity(sa, sb, weights=None)
    
    # Method 3: TF-IDF Weighted Word2Vec
    sim_weighted = cbow_pipeline.sentence_similarity(sa, sb, weights=tfidf_model)
    
    print(f"  -> 1. Sparse TF-IDF Cosine Similarity:       {sim_sparse:+.4f}")
    print(f"  -> 2. Unweighted Word2Vec Average:           {sim_unweighted:+.4f}")
    print(f"  -> 3. TF-IDF Weighted Word2Vec Average:      {sim_weighted:+.4f}")

✓ TFIDFModel fitted with 1,903 terms.

Test Case: Related Concepts (Transformer & Attention)
  Sentence A: 'the transformer relies entirely on an self attention mechanism without recurrence'
  Sentence B: 'multi head attention connects the encoder and decoder representations'
  -> 1. Sparse TF-IDF Cosine Similarity:       +0.0675
  -> 2. Unweighted Word2Vec Average:           +0.9112
  -> 3. TF-IDF Weighted Word2Vec Average:      +0.9050

Test Case: Shared Common Filler Words, Different Topics
  Sentence A: 'we trained the deep model on eight gpus for three days'
  Sentence B: 'the bert language model was evaluated on eleven downstream nlp tasks'
  -> 1. Sparse TF-IDF Cosine Similarity:       +0.0910
  -> 2. Unweighted Word2Vec Average:           +0.9395
  -> 3. TF-IDF Weighted Word2Vec Average:      +0.9250

Test Case: Word Order Inversion
  Sentence A: 'model uses attention instead of recurrent layers'
  Sentence B: 'model uses recurrent layers instead of attention'
  -> 1. Sparse TF

## Conclusion & Next Step: Sentence Transformers (Step 3.3)

1. **TF-IDF Weighted Word2Vec** successfully reduces the influence of generic stopwords, improving semantic focus over unweighted Word2Vec.
2. **However**, it still cannot resolve **word order** or **contextual nuance** because vector addition remains commutative ($A + B = B + A$).
3. In **Step 3.3**, we transition to **Sentence Transformers** (bi-encoders), which compute a holistic, contextualized dense vector for the entire passage at once!